In [0]:
from pyspark.sql.functions import current_timestamp, lit       #'literal'-lets u add a fixed value(like txt, number or none) to every row in dataframe

In [0]:
products_df = spark.table(
    "ecommerce_project.silver.products"
)

product_history_df = (
    products_df
    .select(
        "product_id",
        "product_name",
        "created_at"
    )
    .withColumnRenamed(
        "created_at",
        "product_created_at"
    )
    .withColumn(
        "effective_from",
        current_timestamp()     #Adds a new column named effective_from and sets its value to the exact time this code runs. This marks when this version of the product record becomes active.
    )
    .withColumn(
        "effective_to",
        lit(None).cast("timestamp")   #Adds a new column named effective_to filled with empty (null) values, defined as a timestamp data type. This means the record is currently open-ended and has not expired yet.
    )
    .withColumn(
        "is_current",
        lit(True)        #Adds a flag column named is_current and sets it to True for every row, showing that these are the active records right now.
    )
)

In [0]:
# Write and verify

(
    product_history_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_project.gold.dim_product_history")
)

In [0]:
display(spark.table("ecommerce_project.gold.dim_product_history"))

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, concat, lit, current_timestamp, count, when, sum as spark_sum

In [0]:
#by using (DeltaTable.forName)-Instantiated a deltaTable object pointing to history table. This object will be used later to perform the actual data merge/upsert.

target_table = DeltaTable.forName(
    spark,
    "ecommerce_project.gold.dim_product_history"
)

current_products = spark.table(
    "ecommerce_project.gold.dim_product_history"
)

In [0]:
product_update_df = (
    current_products
    .filter(
        (col("product_id") == 1) &
        (col("is_current") == True)
    )
    .select(
        "product_id",
        concat(
            col("product_name"),
            lit(" - Updated")
        ).alias("product_name"),
        "product_created_at"
    )
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))
)

In [0]:
staged_updates_df = (
    product_update_df
    .withColumn("merge_key", lit(None).cast("int"))
    .unionByName(
        product_update_df.withColumn(
            "merge_key",
            col("product_id")
        )
    )
)

In [0]:
(
    target_table.alias("target")
    .merge(
        staged_updates_df.alias("source"),
        """
        target.product_id = source.merge_key
        AND target.is_current = true
        """
    )
    .whenMatchedUpdate(
        condition="target.product_name <> source.product_name",
        set={
            "effective_to": "source.effective_from",
            "is_current": "false"
        }
    )
    .whenNotMatchedInsert(
        values={
            "product_id": "source.product_id",
            "product_name": "source.product_name",
            "product_created_at": "source.product_created_at",
            "effective_from": "source.effective_from",
            "effective_to": "source.effective_to",
            "is_current": "source.is_current"
        }
    )
    .execute()
)

In [0]:
display(
    spark.table("ecommerce_project.gold.dim_product_history")
    .filter(col("product_id") == 1)
    .orderBy("effective_from")
)

In [0]:
#Validate the SCD type 2 
scd_validation = (
    spark.table("ecommerce_project.gold.dim_product_history")
    .groupBy("product_id")
    .agg(
        count("*").alias("total_versions"),
        spark_sum(
            when(col("is_current") == True, 1).otherwise(0)
        ).alias("current_versions")
    )
)



In [0]:
display(scd_validation)